In [23]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\Administrateur\Documents\M2i\.venv\Scripts\python.exe
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


In [24]:
import pandas as pd
import numpy as np
import json
import time
import os

**Explication**
Cette liste provients des sources officielles pour le calcule de Nutriscore

In [25]:
debug_mode = False

food_parquet = "../../data/food.parquet"
nutriment_parquet = "../../data/nutrients.parquet"
nutriscope_parquet = "../../data/nutriscope.parquet"
category_parquet = "../../data/category.parquet"

nutriments_list = ["fiber", "proteins", "energy", "saturated-fat", "sugars", "salt", "fruits-vegetables-legumes-estimate-from-ingredients"]
nutriments_list_light = ["fiber", "proteins", "energy", "saturated-fat", "sugars", "salt"]

sel_columns = ["nutriments", "code", "product_name", "nutriscore_grade", "brands", "categories_tags"]


In [26]:
off_nutriments_tags_df = pd.read_parquet(food_parquet, columns=sel_columns)

In [27]:
if debug_mode:
    # liste de TOUT les nom de nutriment
    nutriment_names = set()

    for liste in off_nutriments_tags_df["nutriments"]:
        if isinstance(liste, (list, np.ndarray)):
            for nutriment in liste:
                if isinstance(nutriment, dict):
                    name = nutriment.get("name")
                    if name and not (len(name) > 2 and name[2] == "-"):
                        nutriment_names.add(name)

    len(nutriment_names), sorted(nutriment_names)

In [28]:
if debug_mode:
    for liste in off_nutriments_tags_df["nutriments"]:
        if isinstance(liste, (list, np.ndarray)):
            for nutriment in liste:
                if (isinstance(nutriment, dict) and nutriment.get("name") == "fruits-vegetables-legumes-estimate-from-ingredients"):
                    print(nutriment)
                    raise SystemExit


In [29]:
if debug_mode:
    # valeurs différentes rencontrées
    values = set()

    # couples (100g, unit)
    values_units = set()

    for liste in off_nutriments_tags_df["nutriments"]:
        if isinstance(liste, (list, np.ndarray)):
            for nutriment in liste:
                if (isinstance(nutriment, dict) and nutriment.get("name") == "fruits-vegetables-legumes-estimate-from-ingredients" ):
                    values.add(nutriment.get("100g"))
                    values_units.add((nutriment.get("100g"), nutriment.get("unit")))


In [30]:
if debug_mode:
    # for v in values:
    #     print(v)
    for vu in values_units:
        print(vu)


In [ ]:
unit_set = set()

def extraire_depuis_liste(liste_nutriments, nom_nutriment):
    if not isinstance(liste_nutriments, (np.ndarray, list)):
        return None

    for nutriment in liste_nutriments:
        if isinstance(nutriment, dict) and nutriment.get('name') == nom_nutriment:

            value_100g = nutriment.get('100g')
            try:
                value_100g = float(value_100g)
            except (TypeError, ValueError):
                return None

            if nom_nutriment == "fruits-vegetables-legumes-estimate-from-ingredients":
                return value_100g

            unit = nutriment.get('unit')
            if unit == "&#181;g":
                unit = 'µg'
            elif unit == "% vol / *":
                unit = '% vol'
            elif unit == "kJ":
                unit = 'kj'
            elif unit == "":
                unit = None

            unit_set.add(unit)

            if unit == "g":
                return value_100g
            elif unit == "mg":
                return value_100g * 0.001
            elif unit == "µg":
                return value_100g * 0.000001
            elif unit == "kj":
                return value_100g
            else:
                return None

            return value_100g
        
    return None

# Application sur le DataFrame Pandas
off_light_extended_df_fr = off_nutriments_tags_df.copy()
for name in nutriments_list:
    print(f"Generating data for: {name}")
    off_light_extended_df_fr[name] = off_light_extended_df_fr['nutriments'].apply(lambda x: extraire_depuis_liste(x, name))


Generating data for: fiber
Generating data for: proteins
Generating data for: energy
Generating data for: saturated-fat
Generating data for: sugars
Generating data for: salt
Generating data for: fruits-vegetables-legumes-estimate-from-ingredients


In [ ]:

# save data generated in preview cell
off_light_extended_df_fr_light = off_light_extended_df_fr.copy()
off_light_extended_df_fr_light.drop(columns=['nutriments'], inplace=True)
off_light_extended_df_fr_light.to_parquet(nutriment_parquet, index=False)

In [33]:
# load data from nutriments parquet
off_light_extended_df_fr_light = pd.read_parquet(nutriment_parquet)

In [34]:
print(off_light_extended_df_fr_light)
off_light_extended_df_fr_light["fruits-vegetables-legumes-estimate-from-ingredients"].describe()

                  code                                       product_name  \
0        0000101209159  [{'lang': 'main', 'text': 'Véritable pâte à ta...   
1        0000105000011  [{'lang': 'main', 'text': 'Chamomile Herbal Te...   
2        0000105000042  [{'lang': 'main', 'text': 'Lagg's, herbal tea,...   
3        0000105000059  [{'lang': 'main', 'text': 'Linden Flowers Tea'...   
4        0000105000073  [{'lang': 'main', 'text': 'Herbal Tea, Hibiscu...   
...                ...                                                ...   
4636466  5940477600233                                                 []   
4636467  8906009532905                                                 []   
4636468  2516034005082  [{'lang': 'main', 'text': 'unicarm c-ti afumat...   
4636469  5941896400107                                                 []   
4636470       00810098  [{'lang': 'main', 'text': 'Gummies'}, {'lang':...   

        nutriscore_grade            brands  \
0                      e     

count    518069.000000
mean         12.474643
std          26.813634
min          -2.783203
25%           0.000000
50%           0.000000
75%           6.256976
max        1900.067139
Name: fruits-vegetables-legumes-estimate-from-ingredients, dtype: float64

In [35]:
presence = (off_light_extended_df_fr_light[nutriments_list_light].notna().mean().mul(100).sort_values(ascending=False))

presence

proteins         71.589448
sugars           67.537207
saturated-fat    66.242774
salt             62.538728
energy           52.276613
fiber            38.296152
dtype: float64

In [36]:
off_light_extended_df_fr_light["nb_nutriments"] = off_light_extended_df_fr_light[nutriments_list_light].notna().sum(axis=1)
off_light_extended_df_fr_light["nb_nutriments"].value_counts().sort_index()
off_light_extended_df_fr_light["nb_nutriments"].value_counts(normalize=True).sort_index() * 100


nb_nutriments
0    26.882472
1     1.567830
2     2.594430
3     3.210135
4     8.074439
5    36.228092
6    21.442601
Name: proportion, dtype: float64

In [37]:
col = "fruits-vegetables-legumes-estimate-from-ingredients"
off_light_extended_df_fr_light[ (off_light_extended_df_fr_light[col] < 0) | (off_light_extended_df_fr_light[col] > 100) ][["code", col]]

for liste in off_light_extended_df_fr_light[off_light_extended_df_fr_light[col] >= 1900]["product_name"]:
        print(liste)
        if isinstance(liste, (list, np.ndarray)):
            for value in liste:
                print(value)

print(off_light_extended_df_fr_light[off_light_extended_df_fr_light[col] >= 1900])
# print(off_light_extended_df_fr_light["product_name"].dtype)



[{'lang': 'main', 'text': 'Chai Ka’chava'}
 {'lang': 'en', 'text': 'Chai Ka’chava'}]
{'lang': 'main', 'text': 'Chai Ka’chava'}
{'lang': 'en', 'text': 'Chai Ka’chava'}
                  code                                       product_name  \
2019869  0851139005110  [{'lang': 'main', 'text': 'Chai Ka’chava'}, {'...   

        nutriscore_grade   brands categories_tags  fiber   proteins  energy  \
2019869          unknown  Kachava            None   6.45  40.299999     NaN   

         saturated-fat  sugars  salt  \
2019869           4.03    11.3  1.81   

         fruits-vegetables-legumes-estimate-from-ingredients  nb_nutriments  
2019869                                        1900.067139                5  


In [38]:
nutriments_score = [
    "fiber",
    "proteins",
    "energy",
    "saturated-fat",
    "sugars",
    "salt",
]

off_light_extended_df_fr_light["nb_nutriments"] = off_light_extended_df_fr_light[nutriments_score].notna().sum(axis=1)
print(off_light_extended_df_fr_light["nb_nutriments"])

0          5
1          0
2          2
3          0
4          2
          ..
4636466    0
4636467    0
4636468    0
4636469    0
4636470    6
Name: nb_nutriments, Length: 4636471, dtype: int64


In [39]:
candidats = off_light_extended_df_fr_light[
    off_light_extended_df_fr_light[nutriments_list_light].notna().all(axis=1)
    & off_light_extended_df_fr_light["nutriscore_grade"].notna()
]

candidats[["code", "product_name", "nutriscore_grade"]].head(20)

,code,product_name,nutriscore_grade
31,0000236555909,"[{'lang': 'main', 'text': 'Bakers Best, White ...",c
32,0000236598784,"[{'lang': 'main', 'text': 'Bakers Best, Rye Br...",c
60,0000554004509,"[{'lang': 'main', 'text': 'Pain de mie sans gl...",c
64,0000606009841,"[{'lang': 'main', 'text': 'Beignets framboises...",unknown
121,0000790310013,"[{'lang': 'main', 'text': 'Sour Fruit Gummies'...",e
122,0000790310020,"[{'lang': 'main', 'text': 'Jelly Fish'}, {'lan...",c
124,0000790310075,"[{'lang': 'main', 'text': 'Mixed fruit gummies...",d
183,0000901881326,"[{'lang': 'main', 'text': 'Sirop de bleuet'}, ...",unknown
196,0001390000007,"[{'lang': 'main', 'text': 'Espirulina En Compr...",unknown
207,0002000000776,"[{'lang': 'main', 'text': 'Fondant de saumon'}...",unknown


In [40]:
for nut in nutriments_list_light:
    print(f"{nut} - {candidats[candidats["code"] == "0000790310013"][nut]}")

fiber - 121    0.0
Name: fiber, dtype: float64
proteins - 121    5.0
Name: proteins, dtype: float64
energy - 121    1360.0
Name: energy, dtype: float64
saturated-fat - 121    0.0
Name: saturated-fat, dtype: float64
sugars - 121    57.5
Name: sugars, dtype: float64
salt - 121    0.127
Name: salt, dtype: float64


| Points | Énergie (kJ/100g) | Sucres (g/100g) | AG saturés (g/100g) | Sel (g/100g) |
| -----: | ----------------: | --------------: | ------------------: | -----------: |
|      0 |             ≤ 335 |           ≤ 3,4 |                 ≤ 1 |        ≤ 0,2 |
|      1 |             > 335 |           > 3,4 |                 > 1 |        > 0,2 |
|      2 |             > 670 |           > 6,8 |                 > 2 |        > 0,4 |
|      3 |            > 1005 |            > 10 |                 > 3 |        > 0,6 |
|      4 |            > 1340 |            > 14 |                 > 4 |        > 0,8 |
|      5 |            > 1675 |            > 17 |                 > 5 |        > 1,0 |
|      6 |            > 2010 |            > 20 |                 > 6 |        > 1,2 |
|      7 |            > 2345 |            > 24 |                 > 7 |        > 1,4 |
|      8 |            > 2680 |            > 27 |                 > 8 |        > 1,6 |
|      9 |            > 3015 |            > 31 |                 > 9 |        > 1,8 |
|     10 |            > 3350 |            > 34 |                > 10 |        > 2,0 |
|     11 |                 — |            > 37 |                   — |        > 2,2 |
|     12 |                 — |            > 41 |                   — |        > 2,4 |
|     13 |                 — |            > 44 |                   — |        > 2,6 |
|     14 |                 — |            > 48 |                   — |        > 2,8 |
|     15 |                 — |            > 51 |                   — |        > 3,0 |
|     16 |                 — |               — |                   — |        > 3,2 |
|     17 |                 — |               — |                   — |        > 3,4 |
|     18 |                 — |               — |                   — |        > 3,6 |
|     19 |                 — |               — |                   — |        > 3,8 |
|     20 |                 — |               — |                   — |        > 4,0 |


| Score final | Nutri-Score |
| ----------: | :---------: |
|     **≤ 0** |    **A**    |
|   **1 à 2** |    **B**    |
|  **3 à 10** |    **C**    |
| **11 à 18** |    **D**    |
|    **≥ 19** |    **E**    |


In [41]:
# nutriments_list_light = ["fiber", "proteins", "energy", "saturated-fat", "sugars", "salt"]

conf_datas = {}
conf_datas["energy"] = [3350, 3015, 2680, 2345, 2010, 1675, 1340, 1005, 670, 335]
conf_datas["sugars"] = [51, 48, 44, 41, 37, 34, 31, 27, 24, 20, 17, 14, 10, 6.8, 3.4]
conf_datas["saturated-fat"] = [10, 9, 8, 7, 6, 5, 4, 3, 2, 1]
conf_datas["salt"] = [4.0, 3.8, 3.6, 3.4, 3.2, 3.0, 2.8, 2.6, 2.4, 2.2, 2.0, 1.8, 1.6, 1.4, 1.2, 1.0, 0.8, 0.6, 0.4, 0.2]

good_conf_datas = {}
good_conf_datas["fiber"] = [7.4, 6.3, 5.2, 4.1, 3.0]
good_conf_datas["proteins"] = [17, 14, 12, 9.6, 7.2, 4.8, 2.4]

score_grade = [19, 11, 3, 1]
score_grade_tag = ['e', 'd', 'c', 'b']

# all_full = off_light_extended_df_fr_light[off_light_extended_df_fr_light["nb_nutriments"] == 6].copy()
all_full = off_light_extended_df_fr_light[off_light_extended_df_fr_light["nutriscore_grade"] != "unknown"].copy()

def get_points(input_value, tableau):
    if pd.isna(input_value):
        return 0

    for score, v in enumerate(tableau):
        if input_value >= v:
            return len(tableau) - score

    return 0

def get_grade(bad_score, fiber_score, proteins_score):

    if bad_score < 11:
        score = bad_score - fiber_score - proteins_score
    else:
        score = bad_score - fiber_score

    for index, threshold in enumerate(score_grade):
        if score >= threshold:
            return score_grade_tag[index]

    return 'a'

all_full["short_name"] = all_full["product_name"].apply(lambda x: x[0]["text"] if isinstance(x, (list, np.ndarray)) and len(x) > 0 else None)

all_full["bad_score"] = 0

for key in conf_datas:
    all_full["bad_score"] += all_full[key].apply(lambda x: get_points(x, conf_datas[key]))

for key in good_conf_datas:
    all_full[key + "_score"] = all_full[key].apply(lambda x: get_points(x, good_conf_datas[key]))

all_full["my_grade"] = all_full[["bad_score", "fiber_score", "proteins_score"]].apply(
    lambda row: get_grade(row["bad_score"], row["fiber_score"], row["proteins_score"]), axis=1)

print(all_full[["short_name", "proteins_score", "my_grade", "nutriscore_grade", "nb_nutriments"]].head(50))


                                           short_name  proteins_score  \
0   Véritable pâte à tartiner noisettes chocolat noir               3   
10                        Lagg's, dieter's herbal tea               0   
13                               100% Pure Canola Oil               0   
14  Canola Harvest® Original Vegetable Oil Spread Tub               0   
15  Canola harvest, buttery spread, with flaxseed oil               0   
16                               Lithuanian Rye Bread               3   
26            Mehrkomponeneten Protein 90 C6 Haselnuß               7   
28                                The simpsons donuts               2   
30                              Croissants pur beurre               3   
31                           Bakers Best, White Bread               3   
32                             Bakers Best, Rye Bread               3   
33                                            Eclairs               1   
34          Beignets gourmands parfum choco-noisett

In [42]:
print(len(all_full))

1525729


In [43]:
all_full.loc[16, nutriments_list_light]

fiber             3.16000
proteins          9.23000
energy                NaN
saturated-fat     0.00000
sugars           15.40000
salt              0.00246
Name: 16, dtype: float64

In [44]:
print(sel_columns)

['nutriments', 'code', 'product_name', 'nutriscore_grade', 'brands', 'categories_tags']


In [45]:
comparaison = all_full[all_full["nutriscore_grade"].isin(["a", "b", "c", "d", "e"])].copy()
comparaison["same_grade"] = (comparaison["my_grade"] == comparaison["nutriscore_grade"])

print(comparaison["same_grade"].value_counts())
print()
print(comparaison["same_grade"].value_counts(normalize=True) * 100)

same_grade
True     908253
False    472816
Name: count, dtype: int64

same_grade
True     65.764491
False    34.235509
Name: proportion, dtype: float64


In [46]:
grade_value = {"a": 0, "b": 1, "c": 2, "d": 3, "e": 4}

comparaison["grade_distance"] = (comparaison["my_grade"].map(grade_value) - comparaison["nutriscore_grade"].map(grade_value)).abs()

print(comparaison["grade_distance"].value_counts().sort_index())
print()
print(comparaison["grade_distance"].value_counts(normalize=True).sort_index() * 100)

grade_distance
0    908253
1    287539
2    129736
3     44637
4     10904
Name: count, dtype: int64

grade_distance
0    65.764491
1    20.820031
2     9.393883
3     3.232062
4     0.789533
Name: proportion, dtype: float64


In [47]:
all_full.drop(columns=['product_name'], inplace=True)
all_full.to_parquet(nutriscope_parquet, index=False)

In [48]:
all_full["nb_categories"] = all_full["categories_tags"].apply(lambda x: len(x) if isinstance(x, (list, np.ndarray)) else 0)

In [87]:
# print(all_full["nb_categories"].value_counts().head(20))
print(all_full["nb_categories"].describe())
print(all_full["nb_categories"].max())

tmp = all_full.copy()
print(tmp[tmp["nb_categories"] == 5][["short_name", "categories_tags"]])

# for value in tmp[tmp["nb_categories"] == 5][["short_name", "categories_tags"]]["short_name"]:
#     print(value)

# for values in tmp[tmp["nb_categories"] == 5][["short_name", "categories_tags"]]:
#     print(values)
#     break


count    1.525729e+06
mean     5.201278e+00
std      3.164004e+00
min      0.000000e+00
25%      3.000000e+00
50%      5.000000e+00
75%      7.000000e+00
max      7.200000e+01
Name: nb_categories, dtype: float64
72
                                                short_name  \
13                                    100% Pure Canola Oil   
28                                     The simpsons donuts   
38                           Piasten, Chocolate Assortment   
43                                              Dental gum   
59       Lindt williams, liquor chocolate with williams...   
...                                                    ...   
4635451                                       Garam Masala   
4635788                                         Rocky Road   
4635974                                                NaN   
4636050                                Vegane Mini Griller   
4636093          Ultimate Kale Chips - Better Than Cheddar   

                                        

**Réponse aux questions**
- combien de produits vendus en France ?
- quelle part a un Nutri-Score renseigné ?-
- les dix marques les plus présentes ?
- le taux de manquants sur les nutriments clés ( energy_100g , sugars_100g , salt_100g ) ?


In [49]:

def filter_by_country(value, country):
    try:
        # Transforme la chaîne JSON en véritable liste Python
        list_array = list(value)
        return country in list_array
    except (json.JSONDecodeError, TypeError):
        # Gestion des erreurs si le JSON est malformé ou s'il y a un NaN
        return False

off_country_df = pd.read_parquet(food_parquet, columns=["countries_tags"])
mask = off_country_df["countries_tags"].apply(lambda x: filter_by_country(x, "en:france"))

nb_value_france = mask.sum()
print(f"Produits vendus en France : {nb_value_france:,}")

Produits vendus en France : 1,247,336


In [50]:
nutiscore = ['a', 'b', 'c', 'd', 'e']

off_nutriscore_grade_df = pd.read_parquet(food_parquet, columns=["nutriscore_grade", "countries_tags"])

mask_fr = off_nutriscore_grade_df["countries_tags"].apply(lambda x: filter_by_country(x, "en:france"))
mask_nutriscore = off_nutriscore_grade_df["nutriscore_grade"].isin(nutiscore)

tmp = off_nutriscore_grade_df[mask_nutriscore]
tmp_fr = off_nutriscore_grade_df[mask_nutriscore & mask_fr]

print(f"nutriscore présent: {len(tmp)} / {len(tmp) / len(off_nutriscore_grade_df) * 100.0}")
print(f"(FR) nutriscore présent: {len(tmp_fr)} / {len(tmp_fr) / nb_value_france * 100.0}")

nutriscore présent: 1381069 / 29.787072969937693
(FR) nutriscore présent: 463757 / 37.17979758461233


In [51]:
off_brands_df = pd.read_parquet(
    food_parquet,
    columns=["brands", "countries_tags"]
)

print("Top 10 global :")
print(off_brands_df["brands"].value_counts().head(10))

mask_fr_brands = off_brands_df["countries_tags"].apply(
    lambda x: filter_by_country(x, "en:france")
)

tmp_fr = off_brands_df[mask_fr_brands]

print("\nTop 10 France :")
print(tmp_fr["brands"].value_counts().head(10))

Top 10 global :
brands
             108709
Carrefour     20732
Coop          14545
Lidl          14243
U             12384
Aldi          12353
BonÀrea       12155
Hacendado     10658
Auchan        10590
Tesco         10503
Name: count, dtype: int64

Top 10 France :
brands
                54688
Carrefour       12052
U               11974
Auchan           6291
Leader Price     5427
Casino           5119
Cora             3960
Le Gaulois       3548
Picard           3501
Monoprix         3399
Name: count, dtype: int64


In [52]:
nutriments_list_tp = ["energy", "sugars", "salt"]

mask_fr_brands = off_brands_df["countries_tags"].apply(lambda x: filter_by_country(x, "en:france"))
tmp_fr = off_light_extended_df_fr_light[mask_fr_brands]
presence = 100.0 - (tmp_fr[nutriments_list_tp].notna().mean().mul(100).sort_values(ascending=False))

print(presence)

energy    28.798495
sugars    29.387511
salt      33.741991
dtype: float64
